In [2]:
import pandas as pd
import numpy as np
import requests
from urllib.parse import urlparse

# 1. Load the PhiUSIIL Dataset
data_path = "../data/PhiUSIIL_Phishing_URL_Dataset.csv"
df = pd.read_csv(data_path)

print(f"Total Rows Loaded: {df.shape[0]}")
print(f"Total Columns: {df.shape[1]}")
display(df.head(3))

Total Rows Loaded: 235795
Total Columns: 56


,FILENAME,URL,URLLength,Domain,DomainLength,IsDomainIP,TLD,URLSimilarityIndex,CharContinuationRate,TLDLegitimateProb,...,Pay,Crypto,HasCopyrightInfo,NoOfImage,NoOfCSS,NoOfJS,NoOfSelfRef,NoOfEmptyRef,NoOfExternalRef,label
0,521848.txt,https://www.southbankmosaics.com,31,www.southbankmosaics.com,24,0,com,100.0,1.000000,0.522907,...,0,0,1,34,20,28,119,0,124,1
1,31372.txt,https://www.uni-mainz.de,23,www.uni-mainz.de,16,0,de,100.0,0.666667,0.032650,...,0,0,1,50,9,8,39,0,217,1
2,597387.txt,https://www.voicefmradio.co.uk,29,www.voicefmradio.co.uk,22,0,uk,100.0,0.866667,0.028555,...,0,0,1,10,2,7,42,2,5,1


In [3]:
# Known URL shortening domains
SHORTENER_DOMAINS = {
    'bit.ly', 'tinyurl.com', 't.co', 'goo.gl', 'ow.ly', 
    'is.gd', 'buff.ly', 'cutt.ly', 'rb.gy', 'shorturl.at'
}

def resolve_short_url(url, timeout=5):
    """
    Phase 1 Module: Unrolls shortened URLs via HTTP HEAD/GET redirect headers
    and extracts the true destination URL, root domain, and redirect hop count.
    """
    try:
        parsed_initial = urlparse(url)
        initial_domain = parsed_initial.netloc.lower().replace("www.", "")
        is_shortened = initial_domain in SHORTENER_DOMAINS
        
        # Trace HTTP redirects without downloading heavy page content
        response = requests.head(url, allow_redirects=True, timeout=timeout)
        final_url = response.url
        redirect_hops = len(response.history)
        final_domain = urlparse(final_url).netloc.lower().replace("www.", "")
        
        return {
            "Initial_URL": url,
            "Is_Shortener_Service": is_shortened,
            "Resolved_Final_URL": final_url,
            "Extracted_Root_Domain": final_domain,
            "NoOfURLRedirect": redirect_hops
        }
    except Exception as e:
        return {
            "Initial_URL": url,
            "Error": str(e),
            "Resolved_Final_URL": url,
            "NoOfURLRedirect": 0
        }

# Quick verification test (Safe offline/online demo for your guide)
print("Short-URL Redirect Resolution Module initialized successfully.")

Short-URL Redirect Resolution Module initialized successfully.


In [5]:
# 1. Clean the Domain column (lowercase, strip 'www.') to serve as our strict Grouping ID
groups = df['Domain'].astype(str).str.lower().str.replace(r'^www\.', '', regex=True)

# 2. Extract the Target Label (y) -> 1 = Legitimate, 0 = Phishing (in PhiUSIIL)
y = df['label']

# 3. Drop non-numeric/raw text columns and the target label to form our pure numerical Feature Matrix (X)
text_and_target_cols = ['FILENAME', 'URL', 'Domain', 'TLD', 'Title', 'label']
cols_to_drop = [col for col in text_and_target_cols if col in df.columns]

X = df.drop(columns=cols_to_drop)

# 4. Verify Phase 1 Output
print("--- PHASE 1 COMPLETE ---")
print(f"Feature Matrix (X) shape : {X.shape} (Pure numerical features)")
print(f"Target Vector (y) shape  : {y.shape}")
print(f"Unique Root Domains      : {groups.nunique()} out of {len(df)} total URLs")
print(f"\nNotice how {len(df) - groups.nunique()} URLs share duplicate root domains! This proves why random splitting causes leakage.")

--- PHASE 1 COMPLETE ---
Feature Matrix (X) shape : (235795, 50) (Pure numerical features)
Target Vector (y) shape  : (235795,)
Unique Root Domains      : 219103 out of 235795 total URLs

Notice how 16692 URLs share duplicate root domains! This proves why random splitting causes leakage.
